<div style="border-left:6px solid #2f9e41; padding:8px 14px; background:rgba(47,158,65,0.08)">
<b>IFRJ — Campus Paracambi</b> · Engenharia Mecânica<br>
Oficina de Simulação Computacional em Ciências Térmicas · <b>Prof. Luiz Ferreira</b>
</div>

# Dia 2 — Da condução à convecção: a cavidade em partes

Hoje o fluido se move. Vamos montar, peça por peça, uma simulação de **escoamento com transferência de calor**: uma cavidade com a tampa deslizando, um dos problemas-teste mais famosos da dinâmica dos fluidos computacional (CFD).

Como ontem, o código está pronto e comentado: você **lê, roda, prevê, muda e compara**, e em cada parte **escreve** a linha principal numa tarefa ✍️, com a resposta recolhida logo abaixo. No fim, um **desafio em duplas** 🏆.

| Parte | Assunto | Tempo |
|---|---|---|
| A | Advecção–difusão 1D: o calor carregado pelo escoamento | 20 min |
| B | Calor num vórtice conhecido (sem Navier–Stokes ainda) | 25 min |
| C | A pressão como correção: o método da projeção | 20 min |
| D | A cavidade completa, com temperatura | 40 min |

**Extras de bioengenharia:** o Péclet nos vasos sanguíneos (fim da Parte A) e a cavidade como modelo de aneurisma (Experimento D4).

- **Se a sua linha numa tarefa der erro de Python** (um parêntese faltando, por exemplo), o *Executar tudo* para nela. Corrija ou apague a linha, rode a célula 👀 Resposta logo abaixo e continue executando as células seguintes (no Colab, menu *Ambiente de execução*, opção de executar as células depois da atual; no JupyterLite, *Run → Run Selected Cell and All Below*).

Lembretes: execute as células de cima para baixo; o índice (☰) mostra os títulos; nos experimentos, mude os valores da lista `casos` e rode de novo.

In [ ]:
import numpy as np                        # vetores e contas numéricas
import matplotlib.pyplot as plt           # gráficos
import scipy.sparse as sp                 # matrizes esparsas (só guardam os valores não nulos)
from scipy.sparse.linalg import splu      # fatoração LU de matriz esparsa
import time

plt.rcParams.update({"figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3})
print("Bibliotecas carregadas.")

### ⚙️ O verificador das tarefas

Rode a célula abaixo (ela fica recolhida; não precisa abrir). Ela cria as funções `confere_...()`, que conferem o código que você escrever nas **tarefas ✍️** e explicam, em português, o erro mais provável.

In [ ]:
#@title ⚙️ Verificador das tarefas do Dia 2 (rode esta célula; não precisa abrir)
# Cria as funções confere_...(), que testam o código das Tarefas ✍️ e explicam o erro em português.
# Se a sua função passa no teste, os experimentos seguintes usam o SEU código; se não, usam a resposta.
import time
import numpy as np
import matplotlib.pyplot as plt

_SEU, _REF = {}, {}
_FALTA = "⏳ Ainda falta completar: troque o None pela fórmula (mantendo o return) e rode a célula de novo."

def _usar(nome):
    """Função do aluno, se passou no verificador; senão, a resposta."""
    return _SEU.get(nome, _REF[nome])

def _br(valor, formato):
    return format(valor, formato).replace(".", ",")

def _roda(func, *args):
    """Chama a função do aluno e devolve (resultado, problema). problema = None se rodou bem."""
    try:
        res = func(*args)
    except Exception as e:
        m = str(e)
        if "NoneType" in m or "dtype('O')" in m:
            return None, _FALTA
        if "broadcast" in m:
            return None, "✘ As fatias não estão alinhadas (têm tamanhos diferentes). Confira os índices de cada vizinho."
        return None, f"✘ O Python deu um erro ao rodar a sua função: {type(e).__name__}: {e}"
    if res is None or (isinstance(res, tuple) and any(x is None for x in res)):
        return None, _FALTA
    try:
        arr = np.asarray(res, dtype=float)
        if np.any(np.isnan(arr)):
            return None, _FALTA
    except (TypeError, ValueError):
        return None, _FALTA
    return arr, None

def _fim(nome, func, ok, msg, resposta):
    if resposta:
        print(("✔ Esta é a resposta. " if ok else "⚠ ") + msg)
    elif ok:
        _SEU[nome] = func
        print("✔ Certo! " + msg + "\n   Os experimentos abaixo vão usar o SEU código.")
    else:
        _SEU.pop(nome, None)
        print(msg + "\n   Enquanto isso, os experimentos usam a resposta (célula recolhida logo abaixo).")

def _igual(a, b, tol=1e-9):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=1e-9)

# ------------------------------------------------------------------ respostas de referência
def _ref_upwind(T, sigma): return sigma * (T - np.roll(T, 1))
def _ref_v(X, Y, U): return -U * np.cos(np.pi * X) * np.sin(np.pi * Y)
def _ref_correcao(u_prov, v_prov, p, dt, d):
    u, v = u_prov.copy(), v_prov.copy()
    u[1:-1, 1:-1] -= dt * (p[1:-1, 2:] - p[1:-1, :-2]) / (2 * d)
    v[1:-1, 1:-1] -= dt * (p[2:, 1:-1] - p[:-2, 1:-1]) / (2 * d)
    return u, v
_REF.update({"upwind": _ref_upwind, "v_vortice": _ref_v, "correcao": _ref_correcao})

# ------------------------------------------------------------------ Tarefa A
def confere_upwind(func, resposta=False):
    T = np.array([20.0, 20.0, 80.0, 80.0, 20.0]); s = 0.5
    v, problema = _roda(func, T.copy(), s)
    if problema: return _fim("upwind", func, False, problema, resposta)
    certo = _ref_upwind(T, s)
    if _igual(v, certo):
        x = np.arange(200) * 0.01; B = np.where((x >= 0.5) & (x <= 1.0), 80.0, 20.0); Bn = B.copy()
        for _ in range(100):
            Bn = Bn - np.asarray(func(Bn, 1.0), float)
        dif = np.max(np.abs(Bn - np.roll(B, 100)))
        return _fim("upwind", func, dif < 1e-9, "O passo do Exemplo A.1 da apostila deu [20; 20; 50; 80; 50], e com σ = 1 "
                    f"o bolsão andou exatamente uma célula por passo (diferença para o deslocamento exato: {_br(dif, '.1g')}).", resposta)
    if _igual(v, s * (T - np.roll(T, -1))):
        msg = "✘ Você usou o vizinho da DIREITA (np.roll(T, -1)). No upwind, com u > 0, o fluido vem da esquerda: np.roll(T, 1)."
    elif _igual(v, -certo):
        msg = "✘ Sinal trocado: o termo é σ·(T − vizinho da esquerda), nessa ordem."
    elif _igual(v, certo / s):
        msg = "✘ Faltou multiplicar por σ (sigma)."
    elif _igual(v, 0.5 * s * (np.roll(T, -1) - np.roll(T, 1))):
        msg = "✘ Essa é a diferença CENTRAL. A tarefa pede o upwind: σ·(T − np.roll(T, 1))."
    else:
        msg = ("✘ Para T = [20; 20; 80; 80; 20] e σ = 0,5, o termo deveria ser [0; 0; 30; 0; −30]; deu ["
               + "; ".join(_br(t, "g") for t in v) + "].")
    _fim("upwind", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa B
def confere_v_vortice(func, resposta=False):
    n = 41; malha = np.linspace(0, 1, n); X, Y = np.meshgrid(malha, malha); U = 1.0; d = 1 / (n - 1)
    v, problema = _roda(func, X, Y, U)
    if problema: return _fim("v_vortice", func, False, problema, resposta)
    if v.shape != X.shape:
        return _fim("v_vortice", func, False, "✘ v deve ter o mesmo tamanho de X e Y (use X e Y, não números).", resposta)
    u = U * np.sin(np.pi * X) * np.cos(np.pi * Y)
    div = np.abs(u[1:-1, 2:] - u[1:-1, :-2] + v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * d)
    parede = max(np.abs(v[0, :]).max(), np.abs(v[-1, :]).max())
    certo = _ref_v(X, Y, U)
    if _igual(v, certo):
        return _fim("v_vortice", func, True, f"Divergência máxima de {_br(div.max(), '.1e')} (zero, a menos do arredondamento) "
                    "e v = 0 nas paredes de cima e de baixo: o fluido gira sem atravessá-las.", resposta)
    if _igual(v, -certo):
        msg = f"✘ A massa não se conserva (divergência de até {_br(div.max(), '.1f')}): confira o sinal, v = −∂ψ/∂x."
    elif _igual(v, np.pi * certo) or _igual(v, certo / np.pi):
        msg = "✘ Quase: ao derivar sen(πx), o π que sai cancela o 1/π de ψ. Fica v = −U·cos(πx)·sen(πy)."
    elif parede > 1e-9:
        msg = f"✘ O fluido atravessa as paredes de cima e de baixo (v chega a {_br(parede, '.2f')} em y = 0 ou y = 1). Confira onde vão o seno e o cosseno."
    else:
        msg = f"✘ A massa não se conserva: a divergência chega a {_br(div.max(), '.2f')}. Derive ψ = (U/π)·sen(πx)·sen(πy) em x e troque o sinal."
    _fim("v_vortice", func, False, msg, resposta)

# ------------------------------------------------------------------ Tarefa C
def confere_correcao(func, resposta=False):
    g = np.random.default_rng(2); up, vp, p = g.normal(size=(3, 7, 7)); dt, d = 0.3, 0.1
    r, problema = _roda(func, up.copy(), vp.copy(), p, dt, d)
    if problema: return _fim("correcao", func, False, problema, resposta)
    if r.shape != (2,) + up.shape:
        return _fim("correcao", func, False, "✘ A função deve terminar com return u, v.", resposta)
    u, v = r
    cu, cv = _ref_correcao(up, vp, p, dt, d)
    if _igual(u, cu) and _igual(v, cv):
        n = 41; dd = 1 / (n - 1); malha = np.linspace(0, 1, n); X, Y = np.meshgrid(malha, malha)
        uf = np.sin(np.pi * X) * np.cos(np.pi * Y) - 0.2 * np.pi * np.sin(2 * np.pi * X) * np.cos(np.pi * Y)
        vf = -np.cos(np.pi * X) * np.sin(np.pi * Y) - 0.1 * np.pi * np.cos(2 * np.pi * X) * np.sin(np.pi * Y)
        lado = divergente(uf, vf, dd).ravel(); lado -= lado.mean(); lado[0] = 0.0
        pp = np.zeros_like(uf); pp[1:-1, 1:-1] = matriz_pressao(n, dd).solve(lado).reshape(n - 2, n - 2)
        pp[0, :], pp[-1, :] = pp[1, :], pp[-2, :]; pp[:, 0], pp[:, -1] = pp[:, 1], pp[:, -2]
        uc, vc = func(uf, vf, pp, 1.0, dd)
        antes = np.abs(divergente(uf, vf, dd)).max(); depois = np.abs(divergente(uc, vc, dd))[4:-4, 4:-4].max()
        return _fim("correcao", func, depois < 0.05, f"No campo do Experimento C1 (malha 41 × 41), a divergência no interior caiu de "
                    f"{_br(antes, '.2f')} para {_br(depois, '.3f')}.", resposta)
    gx = np.zeros_like(p); gy = np.zeros_like(p)                       # variantes com os erros mais comuns
    gx[1:-1, 1:-1] = (p[1:-1, 2:] - p[1:-1, :-2]); gy[1:-1, 1:-1] = (p[2:, 1:-1] - p[:-2, 1:-1])
    if not (np.array_equal(u[0], up[0]) and np.array_equal(u[-1], up[-1]) and np.array_equal(u[:, 0], up[:, 0])):
        msg = "✘ As bordas mudaram: corrija só os nós internos, u[1:-1, 1:-1] e v[1:-1, 1:-1]."
    elif _igual(u, up + dt * gx / (2 * d)) and _igual(v, vp + dt * gy / (2 * d)):
        msg = "✘ Sinal trocado: a correção é u = u* − Δt·∂p/∂x (subtrair)."
    elif _igual(u, up - dt * gy / (2 * d)) and _igual(v, vp - dt * gx / (2 * d)):
        msg = ("✘ Trocou x e y: em u[j, i] a coluna (2º índice) é x. ∂p/∂x usa p[1:-1, 2:] e p[1:-1, :-2]; "
               "∂p/∂y usa p[2:, 1:-1] e p[:-2, 1:-1].")
    elif _igual(u, up - gx / (2 * d)) and _igual(v, vp - gy / (2 * d)):
        msg = "✘ Faltou multiplicar por Δt (dt)."
    elif _igual(u, up - dt * gx / d) and _igual(v, vp - dt * gy / d):
        msg = "✘ Faltou o 2 da diferença central: divida por 2·d."
    else:
        msg = ("✘ A correção não bate com u = u* − Δt·(p[1:-1, 2:] − p[1:-1, :-2])/(2d) e "
               "v = v* − Δt·(p[2:, 1:-1] − p[:-2, 1:-1])/(2d). Confira cada termo.")
    _fim("correcao", func, False, msg, resposta)

# ------------------------------------------------------------------ Desafio final
_TENTATIVAS = []
def confere_desafio(escolha):
    n, esquema = int(escolha.get("n", 41)), escolha.get("esquema", "central")
    if not 11 <= n <= 81:
        print("Escolha n entre 11 e 81 (malhas maiores demoram demais para a aula)."); return
    if esquema not in ("central", "upwind"):
        print('O esquema deve ser "central" ou "upwind".'); return
    inicio = time.perf_counter()
    res = simula_cavidade(Re=100, n=n, esquema=esquema)
    tempo = time.perf_counter() - inicio
    m = res["malha"]
    u_meio = np.array([np.interp(0.5, m, res["u"][j, :]) for j in range(n)])   # velocidade u na linha x = 0,5
    erro = np.max(np.abs(np.interp(y_ghia, m, u_meio) - u_ghia))
    desvio = 100 * (res["Nu_fundo"] - 1.94) / 1.94
    ok = abs(desvio) <= 3 and erro < 0.03
    _TENTATIVAS.append((n, esquema, res["Nu_fundo"], desvio, erro, tempo, ok))
    print(f"\n{'malha':>7} {'esquema':>8} {'Nu fundo':>9} {'desvio':>8} {'dif. Ghia':>10} {'tempo':>7}")
    validas = [t for t in _TENTATIVAS if t[6]]
    melhor = min(validas, key=lambda t: (t[0], t[5])) if validas else None
    for t in sorted(_TENTATIVAS, key=lambda t: (not t[6], t[0])):
        marca = " ★" if t is melhor else (" ✔" if t[6] else "")
        print(f"{t[0]:4d}×{t[0]:<3d}{t[1]:>8} {_br(t[2], '9.3f')} {_br(t[3], '+7.1f')}% {_br(t[4], '10.3f')} {_br(t[5], '6.1f')} s{marca}")
    if ok:
        print(f"\n✔ Cumpre a meta! Anote no quadro: {n} nós, {esquema}. Dá para cumprir com menos nós?")
    else:
        falta = []
        if abs(desvio) > 3: falta.append(f"o Nusselt está {_br(abs(desvio), '.1f')} % longe de 1,94 (limite: 3 %)")
        if erro >= 0.03: falta.append(f"a diferença para Ghia é {_br(erro, '.3f')} (limite: 0,030)")
        print("\n✘ Ainda não: " + " e ".join(falta) + ". Refine a malha ou troque o esquema.")

print("Verificador pronto: as funções confere_...() estão carregadas.")

## Parte A — Advecção–difusão 1D

### A.1 A equação

Num fluido que escoa com velocidade $u$, a temperatura é **transportada** pelo escoamento (advecção) e **espalhada** pela condução (difusão):

$$\frac{\partial T}{\partial t}+u\,\frac{\partial T}{\partial x}=\alpha\,\frac{\partial^{2}T}{\partial x^{2}}$$

Para o termo de advecção há duas escolhas clássicas: a diferença **central** (2ª ordem) e a diferença ***upwind*** (1ª ordem), que usa o vizinho **de onde vem o escoamento**. Dois números adimensionais decidem o comportamento:

- **Courant:** $\sigma=u\,\Delta t/\Delta x$, quantas células o fluido anda num passo. O método explícito exige $\sigma\le 1$ (condição CFL).
- **Péclet da célula:** $Pe_\Delta=u\,\Delta x/\alpha$, advecção comparada com difusão numa célula.

In [ ]:
def adveccao_difusao_periodica(T0, u, alpha, dx, dt, t_final, esquema="upwind"):
    """dT/dt + u dT/dx = alpha d²T/dx² num domínio periódico: o que sai por um lado entra pelo outro.
    esquema: "upwind" ou "central" para o termo de advecção (u > 0)."""
    sigma = u * dt / dx            # número de Courant
    r = alpha * dt / dx**2         # número de Fourier da malha
    n_passos = int(round(t_final / dt))
    T = T0.copy()
    termo_upwind = _usar("upwind")   # a SUA função da Tarefa A, se passou no verificador; senão, a resposta
    for passo in range(n_passos):
        # np.roll desloca o vetor dando a volta: roll(T, 1) põe em cada posição o valor do vizinho da ESQUERDA
        T_esq = np.roll(T, 1)
        T_dir = np.roll(T, -1)
        if esquema == "upwind":
            adveccao = termo_upwind(T, sigma)             # o fluido vem da esquerda, u > 0 (Tarefa A)
        else:
            adveccao = 0.5 * sigma * (T_dir - T_esq)      # diferença central
        difusao = r * (T_dir - 2 * T + T_esq)
        T = T - adveccao + difusao
    return T, n_passos * dt


def degrau(x, centro_esq=0.5, centro_dir=1.0):
    """Fluido a 20 °C com um bolsão quente a 80 °C entre centro_esq e centro_dir."""
    return np.where((x >= centro_esq) & (x <= centro_dir), 80.0, 20.0)   # np.where(condição, se sim, se não)

print("Motor da Parte A carregado.")

### ✍️ Tarefa A — o termo de advecção upwind

Complete o termo $\sigma\,(T_i-T_{i-1})$ para todos os nós. O domínio é periódico (o que sai por um lado entra pelo outro): use `np.roll(T, 1)`, que põe em cada posição o vizinho da **esquerda**, de onde vem o fluido.

O verificador testa o Exemplo A.1 da apostila e confere se, com $\sigma=1$, o bolsão quente anda exatamente uma célula por passo. Se travar, abra a resposta logo abaixo.

In [ ]:
def minha_adveccao_upwind(T, sigma):
    return None   # ← σ·(T_i − T_(i−1)) para todos os nós; o vizinho da esquerda é np.roll(T, 1)

confere_upwind(minha_adveccao_upwind)

👀 **Resposta da Tarefa A:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa A (abra se travar)
def adveccao_upwind_resposta(T, sigma):
    return sigma * (T - np.roll(T, 1))

confere_upwind(adveccao_upwind_resposta, resposta=True)

### Experimento A1 — a condição CFL

Um bolsão de fluido quente viaja num tubo de 2 m a 1 m/s, sem condução ($\alpha=0$). A solução exata é o bolsão deslocado. Mudamos só o passo de tempo, através de $\sigma$.

**Antes de rodar:** algum caso vai acertar em cheio? Algum vai explodir?

In [ ]:
L, u = 2.0, 1.0
nx = 200
dx = L / nx
x = np.arange(nx) * dx                  # domínio periódico: o ponto x = L é o próprio x = 0
t_final = 0.5

# ===== EXPERIMENTO A1: mude os números de Courant =====
casos = [
    {"nome": "σ = 0,5", "sigma": 0.5},
    {"nome": "σ = 0,9", "sigma": 0.9},
    {"nome": "σ = 1,0", "sigma": 1.0},
    {"nome": "σ = 1,1 (acima do limite)", "sigma": 1.1},
]

fig, eixos = plt.subplots(1, len(casos), figsize=(15, 3.4), sharey=True)
for ax, caso in zip(eixos, casos):
    dt = caso["sigma"] * dx / u                          # o passo de tempo sai de σ
    T, t = adveccao_difusao_periodica(degrau(x), u, 0.0, dx, dt, t_final, "upwind")
    ax.plot(x, degrau(x, 0.5 + u * t, 1.0 + u * t), "k-", lw=1, label="exata")
    ax.plot(x, T, label="upwind")
    ax.set_title(caso["nome"]); ax.set_xlim(0.6, 2.0); ax.set_ylim(0, 100); ax.set_xlabel("x [m]")
eixos[0].set_ylabel("T [°C]"); eixos[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**O que observar.** Com $\sigma=1$ o resultado é exato: o esquema desloca o perfil exatamente uma célula por passo. Com $\sigma<1$ as quinas se arredondam: é a **difusão numérica**, um espalhamento criado pelo método, não pela física. Acima de 1 a simulação explode: a informação física anda mais de uma célula por passo, e o método não a alcança.

### Experimento A2 — Péclet: upwind ou central?

Um canal com fluido entrando a $T=0$ e a saída mantida a $T=1$ (valores adimensionais). Em regime permanente forma-se uma **camada limite** junto à saída, tanto mais fina quanto maior o número de Péclet $Pe=uL/\alpha$. A solução exata é $T(x)=\dfrac{e^{Pe\,x/L}-1}{e^{Pe}-1}$. Usamos uma malha grossa, de 21 nós.

**Antes de rodar:** qual esquema é mais preciso quando a difusão é forte? E quando é fraca?

In [ ]:
def adveccao_difusao_permanente(nx, Pe, esquema, tol=1e-9):
    """Marcha no tempo até o regime permanente, com T = 0 na entrada (x = 0) e T = 1 na saída (x = 1)."""
    u, L = 1.0, 1.0
    alpha = u * L / Pe
    dx = L / (nx - 1)
    dt = 0.5 * min(1 / (u / dx + 2 * alpha / dx**2), 2 * alpha / u**2)   # passo estável para os dois esquemas
    sigma, r = u * dt / dx, alpha * dt / dx**2
    T = np.zeros(nx); T[-1] = 1.0
    for passo in range(500000):
        Tn = T.copy()
        if esquema == "upwind":
            adveccao = sigma * (Tn[1:-1] - Tn[:-2])
        else:
            adveccao = 0.5 * sigma * (Tn[2:] - Tn[:-2])
        T[1:-1] = Tn[1:-1] - adveccao + r * (Tn[2:] - 2 * Tn[1:-1] + Tn[:-2])
        if np.max(np.abs(T - Tn)) < tol:      # parou de mudar: regime permanente
            break
    return np.linspace(0, L, nx), T

# ===== EXPERIMENTO A2: mude os números de Péclet =====
casos = [{"Pe": 5}, {"Pe": 20}, {"Pe": 100}]
nx = 21

fig, eixos = plt.subplots(1, len(casos), figsize=(14, 3.6))
for ax, caso in zip(eixos, casos):
    x_fino = np.linspace(0, 1, 400)
    ax.plot(x_fino, np.expm1(caso["Pe"] * x_fino) / np.expm1(caso["Pe"]), "k-", label="exata")
    for esquema, marca in [("upwind", "o-"), ("central", "s-")]:
        xs, T = adveccao_difusao_permanente(nx, caso["Pe"], esquema)
        ax.plot(xs, T, marca, ms=4, label=esquema)
    ax.set_title(f"Pe = {caso['Pe']}  →  Pe da célula = {caso['Pe']/(nx-1):.2f}"); ax.set_xlabel("x / L")
eixos[0].set_ylabel("T (adimensional)"); eixos[0].legend()
plt.tight_layout(); plt.show()

**O que observar.** Com Péclet da célula até cerca de 2, a central é a mais precisa. Acima disso ela produz **oscilações sem sentido físico** (temperaturas abaixo da mínima), enquanto o *upwind* continua bem-comportado, embora espalhe a camada limite. Regra prática de CFD: refine a malha onde há gradientes fortes, ou use um esquema que respeite a direção do escoamento.

*Para saber mais:* Patankar (1980), cap. 5; Versteeg e Malalasekera, *An Introduction to Computational Fluid Dynamics*, cap. 5.

### A.2 Extra de bioengenharia: o Péclet nos vasos sanguíneos

O sangue é o "fluido de arrefecimento" do corpo. O número de Péclet baseado no diâmetro do vaso, $Pe=u\,d/\alpha$, diz se o sangue **carrega** calor ao longo do vaso (advecção domina) ou **entra em equilíbrio** com o tecido em volta (difusão domina). Os valores abaixo são ordens de grandeza típicas dos livros de fisiologia.

In [ ]:
alpha_sangue = 0.5 / 3.6e6       # k / (ρ c) do sangue ≈ 1,4e-7 m²/s (parecido com o da água)

# ===== Mude os diâmetros d [m] e as velocidades médias u [m/s] =====
vasos = [
    {"nome": "aorta", "d": 25e-3, "u": 0.3},
    {"nome": "artéria média", "d": 4e-3, "u": 0.2},
    {"nome": "arteríola", "d": 30e-6, "u": 5e-3},
    {"nome": "capilar", "d": 8e-6, "u": 0.5e-3},
]
print(f"{'vaso':<15}{'d':>10}{'u [mm/s]':>10}{'Pe = u d / α':>15}")
for vaso in vasos:
    Pe = vaso["u"] * vaso["d"] / alpha_sangue
    diametro = f"{vaso['d']*1000:.0f} mm" if vaso["d"] >= 1e-3 else f"{vaso['d']*1e6:.0f} µm"
    print(f"{vaso['nome']:<15}{diametro:>10}{vaso['u']*1000:10.1f}{Pe:15.3g}")

**O que observar.** Da aorta ao capilar, o Péclet cai mais de um milhão de vezes. Nas artérias ($Pe$ de milhares) o sangue atravessa o tecido quase sem trocar calor: é ele que leva os 37 °C do centro do corpo até as mãos e os pés. Nos capilares ($Pe\ll 1$) a difusão vence e o sangue sai à temperatura do tecido. É a hipótese por trás do termo de perfusão da equação do biocalor (Experimento B5 do Dia 1). Para quem simula: com Péclet tão alto, um esquema central numa malha grossa oscilaria nas artérias, exatamente como no Experimento A2.

## Parte B — Calor num vórtice conhecido

### B.1 A ideia

Antes de calcular o escoamento, vamos **impor** um escoamento simples e ver como ele transporta calor. Usamos um vórtice que gira dentro de um quadrado sem atravessar as paredes:

$$\psi=\frac{U}{\pi}\,\text{sen}(\pi x)\,\text{sen}(\pi y),\qquad u=\frac{\partial\psi}{\partial y}=U\,\text{sen}(\pi x)\cos(\pi y),\qquad v=-\frac{\partial\psi}{\partial x}$$

Partindo de uma **função de corrente** $\psi$, a massa se conserva automaticamente ($\partial u/\partial x+\partial v/\partial y=0$). O $u$ já está no código; o $v$ é a **Tarefa B**. O fundo é quente ($T=1$), o topo é frio ($T=0$) e as laterais são isoladas. Sem escoamento ($U=0$) só há condução, e o perfil é uma reta. Com o vórtice, a equação da energia em 2D é

$$\frac{\partial T}{\partial t}+u\,\frac{\partial T}{\partial x}+v\,\frac{\partial T}{\partial y}=\alpha\,\nabla^{2}T$$

e o calor trocado com a parede quente é medido pelo **número de Nusselt** médio, $Nu=-\overline{\partial T/\partial y}\big|_{y=0}$, que vale exatamente 1 na condução pura.

### B.2 O motor: duas funções que vamos reaproveitar na cavidade

In [ ]:
def laplaciano(f, d):
    """d²f/dx² + d²f/dy² nos nós internos, com o estêncil de 5 pontos do Dia 1 (Δx = Δy = d)."""
    return (f[1:-1, 2:] + f[1:-1, :-2] + f[2:, 1:-1] + f[:-2, 1:-1] - 4 * f[1:-1, 1:-1]) / d**2


def adveccao(f, u, v, d, esquema="upwind"):
    """u df/dx + v df/dy nos nós internos.
    esquema = "central": diferença central (2ª ordem).
    esquema = "upwind": em cada nó, usa a diferença do lado de onde o fluido vem (1ª ordem, robusto)."""
    f_c, u_c, v_c = f[1:-1, 1:-1], u[1:-1, 1:-1], v[1:-1, 1:-1]   # valores nos nós internos
    if esquema == "central":
        dfdx = (f[1:-1, 2:] - f[1:-1, :-2]) / (2 * d)
        dfdy = (f[2:, 1:-1] - f[:-2, 1:-1]) / (2 * d)
    else:
        # diferença para trás (vizinho da esquerda/de baixo) e para frente (vizinho da direita/de cima)
        dfdx_tras, dfdx_frente = (f_c - f[1:-1, :-2]) / d, (f[1:-1, 2:] - f_c) / d
        dfdy_tras, dfdy_frente = (f_c - f[:-2, 1:-1]) / d, (f[2:, 1:-1] - f_c) / d
        dfdx = np.where(u_c > 0, dfdx_tras, dfdx_frente)   # se u > 0 o fluido vem da esquerda: usa "para trás"
        dfdy = np.where(v_c > 0, dfdy_tras, dfdy_frente)
    return u_c * dfdx + v_c * dfdy


def nusselt_medio(T, d, parede="fundo"):
    """Nusselt médio numa parede horizontal, com derivada de 2ª ordem para dentro do domínio."""
    if parede == "fundo":
        dTdy = (-3 * T[0, :] + 4 * T[1, :] - T[2, :]) / (2 * d)
    else:
        dTdy = (3 * T[-1, :] - 4 * T[-2, :] + T[-3, :]) / (2 * d)
    return np.trapezoid(dTdy, dx=d) if hasattr(np, "trapezoid") else np.trapz(dTdy, dx=d)   # média = integral (L = 1)

print("Funções de advecção, difusão e Nusselt carregadas.")

In [ ]:
def calor_no_vortice(n, U, alpha, esquema="upwind", t_max=60.0, tol=1e-5):
    """Temperatura num quadrado unitário com o vórtice imposto. Fundo T = 1, topo T = 0, laterais isoladas."""
    d = 1.0 / (n - 1)
    malha = np.linspace(0, 1, n)
    X, Y = np.meshgrid(malha, malha)
    u = U * np.sin(np.pi * X) * np.cos(np.pi * Y)       # campo de velocidade imposto (não é calculado)
    v = _usar("v_vortice")(X, Y, U)                     # v = −∂ψ/∂x: a SUA função da Tarefa B, se passou; senão, a resposta

    T = 1.0 - Y                                          # começamos pelo perfil de condução pura
    velocidade_max = np.abs(u).max() + np.abs(v).max()
    dt = 0.8 / (velocidade_max / d + 4 * alpha / d**2)   # CFL + difusão em 2D, com 20 % de folga
    t = 0.0
    while t < t_max:
        T_antigo = T.copy()
        T[1:-1, 1:-1] = T_antigo[1:-1, 1:-1] + dt * (-adveccao(T_antigo, u, v, d, esquema)
                                                      + alpha * laplaciano(T_antigo, d))
        T[:, 0] = T[:, 1]                                # laterais isoladas: dT/dx = 0
        T[:, -1] = T[:, -2]
        t += dt
        if np.max(np.abs(T - T_antigo)) / dt < tol:      # parou de mudar: regime permanente
            break
    return {"X": X, "Y": Y, "u": u, "v": v, "T": T, "t": t, "Nu": -nusselt_medio(T, d, "fundo")}

print("Motor da Parte B carregado.")

### ✍️ Tarefa B — a velocidade $v$ a partir da função de corrente

O motor acima já tem $u=\partial\psi/\partial y=U\,\text{sen}(\pi x)\cos(\pi y)$. Falta $v=-\partial\psi/\partial x$, com $\psi=\dfrac{U}{\pi}\,\text{sen}(\pi x)\,\text{sen}(\pi y)$. Derive no papel e escreva $v$ com `np.sin`, `np.cos` e `np.pi`.

O verificador confere se a massa se conserva ($\partial u/\partial x+\partial v/\partial y=0$) e se o fluido não atravessa as paredes. Se travar, abra a resposta logo abaixo.

In [ ]:
def meu_v(X, Y, U):
    return None   # ← v = −∂ψ/∂x, com ψ = (U/π)·sen(πx)·sen(πy)

confere_v_vortice(meu_v)

👀 **Resposta da Tarefa B:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa B (abra se travar)
def v_resposta(X, Y, U):
    return -U * np.cos(np.pi * X) * np.sin(np.pi * Y)

confere_v_vortice(v_resposta, resposta=True)

### Experimento B1 — quanto o vórtice aumenta a troca de calor?

Mantemos $\alpha=0{,}01$ e aumentamos a intensidade $U$ do vórtice. O número de Péclet $Pe=U\,L/\alpha$ compara advecção e difusão.

**Antes de rodar:** o Nusselt vai crescer na mesma proporção que $U$?

In [ ]:
# ===== EXPERIMENTO B1: mude a intensidade U do vórtice =====
casos = [
    {"nome": "sem escoamento", "U": 0.0},
    {"nome": "vórtice fraco", "U": 0.05},
    {"nome": "vórtice médio", "U": 0.2},
    {"nome": "vórtice forte", "U": 1.0},
]
alpha_b = 0.01

fig, eixos = plt.subplots(1, len(casos), figsize=(16, 4))
resultados_b = []
for ax, caso in zip(eixos, casos):
    res = calor_no_vortice(n=41, U=caso["U"], alpha=alpha_b)
    resultados_b.append(res)
    ax.contourf(res["X"], res["Y"], res["T"], levels=np.linspace(0, 1, 21), cmap="coolwarm")
    if caso["U"] > 0:
        ax.streamplot(res["X"], res["Y"], res["u"], res["v"], color="k", density=0.8, linewidth=0.6)
    ax.set_title(f"{caso['nome']}\nPe = {caso['U']/alpha_b:.0f}, Nu = {res['Nu']:.2f}", fontsize=10)
    ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

**O que observar.** Sem escoamento as isotermas são horizontais e $Nu=1$. O vórtice sobe fluido quente por um lado e desce fluido frio pelo outro: as isotermas se enrolam, ficam espremidas junto às paredes (camadas limite) e o Nusselt cresce. Mas cresce **menos** que $U$: dobrar a velocidade não dobra a troca de calor. É a mesma física das correlações $Nu\propto Pe^{n}$, com $n<1$, dos livros de transferência de calor.

🎥 **No projetor:** o professor troca o esquema do vórtice forte para `"central"` (argumento `esquema` em `calor_no_vortice`). O Nusselt muda muito? E a aparência das isotermas?

## Parte C — A pressão como correção: o método da projeção

### C.1 A ideia

Num escoamento calculado (não imposto), a conservação de massa $\nabla\cdot\vec u=0$ não é garantida pelas equações de movimento. O **método da projeção** (Chorin, 1968) resolve isso em duas etapas:

1. calcula uma velocidade **provisória** $\vec u^{\,*}$ sem se preocupar com a massa;
2. **corrige** essa velocidade com o gradiente de uma pressão: $\vec u=\vec u^{\,*}-\Delta t\,\nabla p$.

Para a velocidade corrigida ter divergência nula, a pressão precisa satisfazer uma equação de **Poisson**, a mesma do regime permanente do Dia 1 com um termo-fonte:

$$\nabla^{2}p=\frac{1}{\Delta t}\,\nabla\cdot\vec u^{\,*},\qquad\frac{\partial p}{\partial n}=0\ \text{nas paredes}$$

Vamos ver essa correção funcionando sozinha, num campo que **não** conserva massa.

In [ ]:
def matriz_pressao(n, d):
    """Matriz do Laplaciano nos nós internos com dp/dn = 0 nas paredes, já fatorada (LU).
    A fatoração é feita UMA vez; depois, cada solução custa só uma substituição."""
    m = n - 2
    D = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(m, m), format="lil")
    D[0, 0] = -1.0       # parede: o vizinho de fora vale o mesmo que o de dentro (dp/dn = 0),
    D[-1, -1] = -1.0     # então "-2 + 1" vira "-1" na primeira e na última linha
    D = D.tocsr() / d**2
    A = (sp.kron(sp.identity(m), D) + sp.kron(D, sp.identity(m))).tolil()   # 2D = soma das direções
    A[0, :] = 0.0        # com dp/dn = 0 em todas as paredes, p só é definida a menos de uma constante:
    A[0, 0] = 1.0        # fixamos p = 0 no primeiro nó para o sistema ter solução única
    return splu(A.tocsc())


def divergente(u, v, d):
    """du/dx + dv/dy nos nós internos (diferenças centrais)."""
    return (u[1:-1, 2:] - u[1:-1, :-2] + v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * d)


def projeta(u_prov, v_prov, d, fatorada, dt=1.0):
    """Corrige a velocidade provisória para que conserve massa (divergência ≈ 0)."""
    m = u_prov.shape[0] - 2
    lado_direito = (divergente(u_prov, v_prov, d) / dt).ravel()
    lado_direito -= lado_direito.mean()   # compatibilidade: com dp/dn = 0, a fonte precisa somar zero
    lado_direito[0] = 0.0                 # o nó onde fixamos p = 0
    p = np.zeros_like(u_prov)
    p[1:-1, 1:-1] = fatorada.solve(lado_direito).reshape(m, m)
    p[0, :], p[-1, :] = p[1, :], p[-2, :]          # dp/dn = 0 nas paredes de baixo e de cima
    p[:, 0], p[:, -1] = p[:, 1], p[:, -2]          # e nas laterais
    u, v = _usar("correcao")(u_prov, v_prov, p, dt, d)   # u = u* − dt ∇p: a SUA função da Tarefa C, se passou; senão, a resposta
    return u, v, p

print("Motor da projeção carregado.")

### ✍️ Tarefa C — a correção pela pressão

Complete as duas linhas da correção $\vec u=\vec u^{\,*}-\Delta t\,\nabla p$ nos nós internos, com diferenças centrais. Lembre a indexação: em `u[j, i]` a **coluna** (2º índice) é $x$ e a **linha** (1º índice) é $y$.

O verificador testa a correção num campo pequeno e depois no campo com fontes do Experimento C1. Se travar, abra a resposta logo abaixo.

In [ ]:
def minha_correcao(u_prov, v_prov, p, dt, d):
    u, v = u_prov.copy(), v_prov.copy()
    u[1:-1, 1:-1] -= None   # ← dt × ∂p/∂x: diferença central com p[1:-1, 2:] e p[1:-1, :-2]
    v[1:-1, 1:-1] -= None   # ← dt × ∂p/∂y: agora as LINHAS vizinhas, p[2:, 1:-1] e p[:-2, 1:-1]
    return u, v

confere_correcao(minha_correcao)

👀 **Resposta da Tarefa C:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Resposta da Tarefa C (abra se travar)
def correcao_resposta(u_prov, v_prov, p, dt, d):
    u, v = u_prov.copy(), v_prov.copy()
    u[1:-1, 1:-1] -= dt * (p[1:-1, 2:] - p[1:-1, :-2]) / (2 * d)
    v[1:-1, 1:-1] -= dt * (p[2:, 1:-1] - p[:-2, 1:-1]) / (2 * d)
    return u, v

confere_correcao(correcao_resposta, resposta=True)

### Experimento C1 — corrigindo um campo que não conserva massa

Somamos ao vórtice da Parte B um campo de "fontes e sumidouros" (que cria e destrói massa). A projeção deve remover essa parte e devolver o vórtice original.

**Antes de rodar:** a divergência vai a zero exatamente? O que acontece ao refinar a malha?

In [ ]:
def campo_com_fontes(n, intensidade=0.1):
    malha = np.linspace(0, 1, n)
    X, Y = np.meshgrid(malha, malha)
    u_vortice = np.sin(np.pi * X) * np.cos(np.pi * Y)          # conserva massa
    v_vortice = -np.cos(np.pi * X) * np.sin(np.pi * Y)
    # Fontes e sumidouros: gradiente de φ = intensidade·cos(2πx)·cos(πy). Cria e destrói massa,
    # mas não atravessa as paredes (componente normal nula nas bordas), como o vórtice.
    u_fontes = -2 * np.pi * intensidade * np.sin(2 * np.pi * X) * np.cos(np.pi * Y)
    v_fontes = -np.pi * intensidade * np.cos(2 * np.pi * X) * np.sin(np.pi * Y)
    return X, Y, u_vortice, v_vortice, u_vortice + u_fontes, v_vortice + v_fontes

# ===== EXPERIMENTO C1: mude as malhas =====
malhas = [11, 21, 41, 81]
print(f"{'malha':>7}{'div. antes':>12}{'div. depois (interior)':>24}{'div. depois (junto às paredes)':>32}{'erro no vórtice':>17}")
for n_teste in malhas:
    d = 1 / (n_teste - 1)
    X, Y, uv, vv, u_prov, v_prov = campo_com_fontes(n_teste)
    u_c, v_c, p = projeta(u_prov, v_prov, d, matriz_pressao(n_teste, d))
    div_antes = np.abs(divergente(u_prov, v_prov, d)).max()
    div_depois = np.abs(divergente(u_c, v_c, d))           # divergência em cada nó interno
    k = max(1, (n_teste - 1) // 10)                          # "interior" = longe 10 % das paredes
    erro = max(np.abs(u_c - uv)[1:-1, 1:-1].max(), np.abs(v_c - vv)[1:-1, 1:-1].max())
    print(f"{n_teste:4d}×{n_teste:<3d}{div_antes:12.2f}{div_depois[k:-k, k:-k].max():24.4f}"
          f"{div_depois.max():32.2f}{erro:17.4f}")

# Figura para a malha de 41 nós: antes, pressão e depois
n = 41; d = 1 / (n - 1)
X, Y, uv, vv, u_prov, v_prov = campo_com_fontes(n)
u_c, v_c, p = projeta(u_prov, v_prov, d, matriz_pressao(n, d))
fig, eixos = plt.subplots(1, 3, figsize=(14, 4.2))
limite = np.abs(divergente(u_prov, v_prov, d)).max()
for ax, (titulo, uu, vv_, campo, mapa) in zip(eixos, [
        ("antes: com fontes e sumidouros", u_prov, v_prov, None, None),
        ("pressão que faz a correção", None, None, p, "viridis"),
        ("depois: só o vórtice", u_c, v_c, None, None)]):
    if campo is not None:
        cf = ax.contourf(X, Y, campo, levels=20, cmap=mapa); fig.colorbar(cf, ax=ax, shrink=0.8)
    else:
        div = np.zeros_like(X); div[1:-1, 1:-1] = divergente(uu, vv_, d)
        ax.contourf(X, Y, div, levels=np.linspace(-limite, limite, 21), cmap="RdBu_r")
        ax.streamplot(X, Y, uu, vv_, color="k", density=0.9, linewidth=0.6)
    ax.set_title(titulo, fontsize=10); ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

**O que observar.** Antes da correção, as linhas de corrente nascem e morrem nas manchas coloridas (fontes e sumidouros); depois, só sobra o vórtice. No interior, a divergência cai cerca de 4 vezes a cada refinamento (2ª ordem) e o erro em relação ao vórtice original cai pela metade. Junto às paredes sobra um resíduo que não diminui: este código trata a parede da forma mais simples possível ($p$ da parede igual ao do vizinho), o que é suficiente para a cavidade, mas não para uma conservação de massa perfeita. Os programas comerciais tratam melhor as paredes e usam técnicas como a **interpolação de Rhie–Chow**, que também evitam campos de pressão "em xadrez".

*Para saber mais:* Chorin (1968); Ferziger, Perić e Street, *Computational Methods for Fluid Dynamics*, cap. 7.

## Parte D — A cavidade completa, com temperatura

### D.1 Juntando as peças

Uma cavidade quadrada cheia de fluido tem a tampa deslizando para a direita. A tampa é quente ($\theta=1$), o fundo é frio ($\theta=0$) e as laterais são isoladas. Em variáveis adimensionais:

$$\frac{\partial\vec u}{\partial t}+(\vec u\cdot\nabla)\vec u=-\nabla p+\frac{1}{Re}\,\nabla^{2}\vec u,\qquad\frac{\partial\theta}{\partial t}+\vec u\cdot\nabla\theta=\frac{1}{Re\,Pr}\,\nabla^{2}\theta,\qquad\nabla\cdot\vec u=0$$

com $Re=UL/\nu$ (Reynolds) e $Pr=\nu/\alpha$ (Prandtl). Repare: $u$, $v$ e $\theta$ obedecem à mesma "advecção + difusão" da Parte B, e a conservação de massa é garantida pela projeção da Parte C. Cada passo de tempo faz:

1. **velocidade provisória:** advecção + difusão de $u$ e $v$ (funções da Parte B);
2. **projeção:** Poisson da pressão e correção (Parte C);
3. **energia:** advecção + difusão de $\theta$ com a velocidade nova.

In [ ]:
def simula_cavidade(Re=100, Pr=0.71, n=41, esquema="central", Ri=0.0, t_max=60.0, tol=1e-4):
    """Cavidade com tampa móvel (u = 1 em y = 1), tampa quente (θ = 1), fundo frio (θ = 0), laterais isoladas.
    Ri = número de Richardson do empuxo (0 = convecção forçada pura)."""
    nu, alpha = 1 / Re, 1 / (Re * Pr)
    d = 1 / (n - 1)
    fatorada = matriz_pressao(n, d)              # fatorada uma vez, usada em todos os passos

    u = np.zeros((n, n)); v = np.zeros((n, n)); T = np.zeros((n, n))
    u[-1, :] = 1.0                               # a tampa se move com velocidade 1
    T[-1, :] = 1.0                               # tampa quente (o fundo, T[0, :], fica em 0)

    t, passos, inicio = 0.0, 0, time.perf_counter()
    while t < t_max:
        dt = 0.8 / ((np.abs(u).max() + np.abs(v).max()) / d + 4 * max(nu, alpha) / d**2)
        u_antigo, T_antigo = u.copy(), T.copy()

        # 1) velocidade provisória: advecção + difusão (+ empuxo na direção vertical, se Ri > 0)
        u_prov, v_prov = u.copy(), v.copy()
        u_prov[1:-1, 1:-1] += dt * (-adveccao(u, u, v, d, esquema) + nu * laplaciano(u, d))
        v_prov[1:-1, 1:-1] += dt * (-adveccao(v, u, v, d, esquema) + nu * laplaciano(v, d)
                                    + Ri * (T[1:-1, 1:-1] - 0.5))
        # 2) projeção: pressão e correção (as paredes mantêm u = v = 0 e a tampa u = 1)
        u, v, p = projeta(u_prov, v_prov, d, fatorada, dt)
        # 3) energia com a velocidade nova
        T[1:-1, 1:-1] = T_antigo[1:-1, 1:-1] + dt * (-adveccao(T_antigo, u, v, d, esquema)
                                                      + alpha * laplaciano(T_antigo, d))
        T[:, 0], T[:, -1] = T[:, 1], T[:, -2]   # laterais isoladas

        t += dt; passos += 1
        if passos % 100 == 0:
            mudanca = max(np.abs(u - u_antigo).max(), np.abs(T - T_antigo).max()) / dt
            if mudanca < tol:                    # regime permanente
                break

    malha = np.linspace(0, 1, n)
    X, Y = np.meshgrid(malha, malha)
    print(f"Re = {Re}, Pr = {Pr}, malha {n}×{n}, {esquema}: t = {t:.1f}, {passos} passos, "
          f"{time.perf_counter() - inicio:.1f} s")
    return {"X": X, "Y": Y, "u": u, "v": v, "p": p, "T": T, "malha": malha,
            "Nu_fundo": nusselt_medio(T, d, "fundo"), "Nu_tampa": nusselt_medio(T, d, "tampa")}

# Dados de referência de Ghia, Ghia e Shin (1982), Re = 100: velocidade u na linha vertical central
y_ghia = np.array([1.0, 0.9766, 0.9688, 0.9609, 0.9531, 0.8516, 0.7344, 0.6172, 0.5,
                   0.4531, 0.2813, 0.1719, 0.1016, 0.0703, 0.0625, 0.0547, 0.0])
u_ghia = np.array([1.0, 0.84123, 0.78871, 0.73722, 0.68717, 0.23151, 0.00332, -0.13641, -0.20581,
                   -0.21090, -0.15662, -0.10150, -0.06434, -0.04775, -0.04192, -0.03717, 0.0])
print("Motor da cavidade carregado.")

### Experimento D1 — o número de Reynolds e a forma do vórtice

**Antes de rodar:** com $Re$ maior (menos viscosidade relativa), o centro do vórtice fica mais perto da tampa ou do centro da cavidade?

In [ ]:
# ===== EXPERIMENTO D1: mude Re (e o esquema: acima de Re ≈ 150 nesta malha, use "upwind") =====
casos = [
    {"Re": 10, "esquema": "central"},
    {"Re": 100, "esquema": "central"},
    {"Re": 400, "esquema": "upwind"},
]

fig, eixos = plt.subplots(1, len(casos), figsize=(15, 4.4))
for ax, caso in zip(eixos, casos):
    res = simula_cavidade(Re=caso["Re"], esquema=caso["esquema"])
    rapidez = np.sqrt(res["u"]**2 + res["v"]**2)            # módulo da velocidade
    cf = ax.contourf(res["X"], res["Y"], rapidez, levels=20, cmap="viridis")
    ax.streamplot(res["X"], res["Y"], res["u"], res["v"], color="w", density=1.1, linewidth=0.7)
    ax.set_title(f"Re = {caso['Re']} ({caso['esquema']})"); ax.set_aspect("equal")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.colorbar(cf, ax=eixos, shrink=0.8, label="|velocidade| / U")
plt.show()

**O que observar.** Com $Re=10$ o escoamento é quase simétrico entre esquerda e direita, e o centro do vórtice fica no meio da largura, perto da tampa ($x\approx 0{,}52$; $y\approx 0{,}75$). Aumentando $Re$, a inércia "empurra" o centro do vórtice para a direita e, depois, para baixo, e surgem pequenos vórtices secundários nos cantos inferiores. Por que o *upwind* em $Re=400$? Nesta malha o Reynolds da célula seria $400\times 0{,}025=10$, bem acima de 2: a central oscilaria (Experimento A2).

### Experimento D2 — ar ou água? (número de Prandtl)

🎥 **No projetor:** o professor roda este experimento e a turma comenta. Se sobrar tempo, rode você também.

**Antes de rodar:** a água ($Pr\approx 7$) difunde calor mais devagar que quantidade de movimento. As isotermas ficam mais "arrastadas" pelo vórtice ou mais retas?

In [ ]:
# ===== EXPERIMENTO D2: mude o Prandtl (para Pr alto, a energia precisa de "upwind" nesta malha) =====
casos = [
    {"nome": "ar", "Pr": 0.71, "esquema": "central"},
    {"nome": "água", "Pr": 7.0, "esquema": "upwind"},
]

fig, eixos = plt.subplots(1, len(casos), figsize=(11, 4.4))
for ax, caso in zip(eixos, casos):
    res = simula_cavidade(Re=100, Pr=caso["Pr"], esquema=caso["esquema"])
    cf = ax.contourf(res["X"], res["Y"], res["T"], levels=np.linspace(0, 1, 21), cmap="inferno")
    ax.streamplot(res["X"], res["Y"], res["u"], res["v"], color="w", density=0.8, linewidth=0.6)
    ax.set_title(f"{caso['nome']}: Pr = {caso['Pr']}, Nu (fundo) = {res['Nu_fundo']:.2f}")
    ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.colorbar(cf, ax=eixos, shrink=0.8, label="θ")
plt.show()

**O que observar.** Com a água, o calor quase não se difunde e é levado pelo vórtice: as isotermas acompanham as linhas de corrente, formam camadas finas junto às paredes e o Nusselt cresce. Atenção ao interpretar: o Nusselt é adimensional. O coeficiente de convecção é $h=Nu\,k/L$, e a condutividade da água é cerca de 23 vezes a do ar. Nusselt maior **e** $k$ muito maior: por isso a água é um fluido de arrefecimento tão melhor que o ar.

### Experimento D3 — a malha é suficiente? (validação)

Comparamos a velocidade na linha vertical central com os dados de Ghia, Ghia e Shin (1982) e o Nusselt médio com o valor de Iwatsu e colaboradores (1993), cerca de 1,94 para $Re=100$.

**Antes de rodar:** a malha de 21 nós será suficiente?

In [ ]:
# ===== EXPERIMENTO D3: mude as malhas (61 já leva alguns segundos) =====
casos = [{"n": 21}, {"n": 41}, {"n": 61}]

fig, ax = plt.subplots(figsize=(5, 5.5))
ax.plot(u_ghia, y_ghia, "ko", mfc="none", ms=6, label="Ghia et al. (1982)")
print(f"{'malha':>7}{'erro máx. vs Ghia':>19}{'Nu fundo':>10}{'Nu tampa':>10}")
for caso in casos:
    res = simula_cavidade(Re=100, n=caso["n"], esquema="central")
    meio = caso["n"] // 2
    u_central = res["u"][:, meio]                                  # coluna do meio: x = 0,5
    erro = np.max(np.abs(np.interp(y_ghia, res["malha"], u_central) - u_ghia))
    print(f"{caso['n']:4d}×{caso['n']:<3d}{erro:17.3f}{res['Nu_fundo']:10.2f}{res['Nu_tampa']:10.2f}")
    ax.plot(u_central, res["malha"], label=f"malha {caso['n']}×{caso['n']}")
ax.set_xlabel("u / U em x = 0,5"); ax.set_ylabel("y / L"); ax.legend()
plt.show()

**O que observar.** A diferença para Ghia cai ao refinar a malha, e o Nusselt da parede fria se aproxima de 1,94. Os Nusselt da tampa e do fundo deveriam ser iguais em regime permanente (o calor que entra sai), mas diferem: os cantos da tampa, onde a velocidade salta de 1 para 0, são singulares e exigem malha muito fina. É a postura certa diante de qualquer simulação: **refinar até os resultados de interesse pararem de mudar e comparar com uma referência confiável**.

**Sua vez (desafio):** em `simula_cavidade`, use `Ri=1.0` (empuxo). Com a tampa quente, o fluido quente fica em cima: o empuxo estabiliza ou desestabiliza o escoamento? Compare o Nusselt.

### 🏆 Desafio final, em duplas — a menor malha que passa na validação

**Meta**, com $Re=100$: Nusselt médio no fundo a menos de **3 %** de 1,94 **e** diferença máxima para Ghia et al. **menor que 0,030**. Ganha a dupla que cumprir a meta com **menos nós**. Mude `n` e `esquema` e rode quantas vezes quiser: a tabela guarda todas as tentativas e marca a melhor com ★. Anotem a melhor no quadro.

**Antes de rodar:** quem precisa de menos nós para a mesma precisão, o central ou o upwind? Por quê?

In [ ]:
# ===== DESAFIO: escolha a malha (n entre 11 e 81) e o esquema ("central" ou "upwind") =====
minha_escolha = {"n": 31, "esquema": "upwind"}

confere_desafio(minha_escolha)

👀 **Uma resposta do desafio:** a célula abaixo está recolhida. Abra (clique nela ou em *Mostrar código*) só se travar, e rode.

In [ ]:
#@title 👀 Uma resposta do desafio (abra se travar)
# Com o esquema central, a menor malha que cumpre a meta tem 38 nós (com 37, o Nusselt fica 3,2 % abaixo de 1,94). Com o upwind, são necessários cerca de 49 nós: o esquema de 2ª ordem precisa de muito menos
# nós para a mesma precisão, desde que o Reynolds da célula não passe muito de 2 (Experimento A2).
confere_desafio({"n": 38, "esquema": "central"})

### Experimento D4 — extra de bioengenharia: a cavidade como aneurisma

Um **aneurisma** é uma dilatação em forma de bolsa na parede de uma artéria. O sangue que passa pela artéria arrasta o sangue da boca da bolsa, como a tampa arrasta o fluido da cavidade: lá dentro forma-se um vórtice lento. A cavidade com tampa móvel é o modelo mais simples desse escoamento. Em artérias cerebrais, o Reynolds é da ordem de algumas centenas.

Duas grandezas interessam aos médicos e engenheiros: as **zonas de sangue quase parado**, onde coágulos (trombos) tendem a se formar, e a **tensão de cisalhamento na parede**, $\tau_p=\mu\,\partial u/\partial y$, que as células da parede do vaso "sentem".

**Antes de rodar:** com Reynolds maior, a região de sangue quase parado no fundo da bolsa aumenta ou diminui?

In [ ]:
# ===== EXPERIMENTO D4: mude Re ou o limite que define "quase parado" =====
casos = [
    {"Re": 100, "esquema": "central"},
    {"Re": 400, "esquema": "upwind"},
]
limite = 0.05        # "quase parado": rapidez abaixo de 5 % da velocidade da artéria

fig, eixos = plt.subplots(1, len(casos) + 1, figsize=(15, 4.4))
for ax, caso in zip(eixos, casos):
    res = simula_cavidade(Re=caso["Re"], esquema=caso["esquema"])
    rapidez = np.sqrt(res["u"]**2 + res["v"]**2)
    parado = rapidez < limite                                  # matriz de True/False
    d = res["malha"][1]
    # cisalhamento adimensional na parede do fundo: du/dy com a fórmula de 2ª ordem usada no Nusselt
    cisalhamento = (-3 * res["u"][0, :] + 4 * res["u"][1, :] - res["u"][2, :]) / (2 * d)
    cf = ax.contourf(res["X"], res["Y"], rapidez, levels=np.linspace(0, 1, 21), cmap="viridis")
    ax.contour(res["X"], res["Y"], rapidez, levels=[limite], colors="red", linewidths=2)   # fronteira do "quase parado"
    ax.streamplot(res["X"], res["Y"], res["u"], res["v"], color="w", density=0.9, linewidth=0.6)
    ax.set_title(f"Re = {caso['Re']}: {100 * parado.mean():.0f} % da bolsa quase parada", fontsize=10)
    ax.set_aspect("equal"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    eixos[-1].plot(res["malha"], np.abs(cisalhamento), label=f"Re = {caso['Re']}")
    cisalhamento_boca = (3 * res["u"][-1, 5:-5] - 4 * res["u"][-2, 5:-5] + res["u"][-3, 5:-5]) / (2 * d)   # sem os cantos
    print(f"   cisalhamento médio: {np.abs(cisalhamento).mean():.2f} U/L no fundo e "
          f"{np.abs(cisalhamento_boca).mean():.1f} U/L na boca da bolsa")
eixos[-1].set_xlabel("x / L ao longo do fundo da bolsa"); eixos[-1].set_ylabel("|∂u/∂y| no fundo [U/L]")
eixos[-1].set_title("Cisalhamento na parede do fundo", fontsize=10); eixos[-1].legend()
plt.tight_layout(); plt.show()

**O que observar.** A linha vermelha é a fronteira da região quase parada (rapidez abaixo de 5 % da velocidade da artéria): ela ocupa o fundo e os cantos da bolsa, longe da boca (e o miolo do vórtice, que gira devagar), e cresce um pouco com o Reynolds (de cerca de 25 % para 30 % da área). É onde o sangue se renova devagar. O cisalhamento no fundo é de 40 a 100 vezes menor que junto à boca: paredes com cisalhamento muito baixo e sangue estagnado são associadas, nos estudos de hemodinâmica computacional, à formação de trombos e à degeneração da parede (cisalhamento muito alto também aparece ligado ao crescimento de aneurismas; a questão segue em aberto: Meng et al., 2014). Hoje simulações desse tipo, em geometrias obtidas de tomografia do próprio paciente, ajudam a avaliar o risco de ruptura e a projetar *stents* que desviam o fluxo.

**Limites do modelo.** O sangue real pulsa, a geometria é tridimensional, a parede é elástica e o sangue não é exatamente newtoniano. A cavidade só captura a ideia central: um vórtice lento movido pelo escoamento que passa na boca.

**Sua vez:** mude `limite` para 0,02. A fração quase parada muda muito? E com $Re=10$ (esquema `"central"`)?

## Fechamento do Dia 2

| Peça | O que você viu | Nos programas comerciais |
|---|---|---|
| Advecção | CFL, upwind × central, difusão numérica | esquemas de 1ª e 2ª ordem, *upwind* de 2ª ordem, controle de Courant |
| Calor no escoamento | Nusselt cresce com Péclet | coeficientes de convecção calculados |
| Projeção | a pressão corrige a velocidade | algoritmos SIMPLE, PISO e acoplados |
| Cavidade | validação e independência de malha | estudo de convergência de malha (GCI) |

### Para saber mais

- Patankar, S. V. *Numerical Heat Transfer and Fluid Flow*. Hemisphere, 1980.
- Versteeg, H. K.; Malalasekera, W. *An Introduction to Computational Fluid Dynamics: The Finite Volume Method*. 2. ed. Pearson, 2007.
- Ferziger, J. H.; Perić, M.; Street, R. L. *Computational Methods for Fluid Dynamics*. 4. ed. Springer, 2020.
- Maliska, C. R. *Transferência de Calor e Mecânica dos Fluidos Computacional*. LTC.
- Fortuna, A. O. *Técnicas Computacionais para Dinâmica dos Fluidos*. EDUSP.
- Chorin, A. J. Numerical solution of the Navier–Stokes equations. *Mathematics of Computation*, 22, 1968.
- Ghia, U.; Ghia, K. N.; Shin, C. T. *Journal of Computational Physics*, 48, 1982.
- Iwatsu, R.; Hyun, J. M.; Kuwahara, K. *International Journal of Heat and Mass Transfer*, 36, 1993.
- Meng, H. et al. High WSS or low WSS? Complex interactions of hemodynamics with intracranial aneurysm initiation, growth, and rupture. *American Journal of Neuroradiology*, 35, 2014.
- Pennes, H. H. Analysis of tissue and arterial blood temperatures in the resting human forearm. *Journal of Applied Physiology*, 1, 1948.
- Inspiração da oficina: L. A. Barba e G. F. Forsyth, *CFD Python* (2018).